# Maia Corpus v8 — Production Resume v2

**Continue the 650 remaining chunks using the recovered SQLite v5.** Based on the original v1.1 full-production notebook. No reprocessing of committed chunks. Existing shards and historical state are preserved.

**Critical:** Source files reside in `quarantine_pending_rights_review`. Processing is **disabled** until you explicitly confirm authorization for the selected corpus. This notebook does not establish licensing. Run one Colab session at a time; do not run the old notebook concurrently. No replacement of the original `state.sqlite3`.

**Execution:** run setup cells, inspect the readiness report, then explicitly set the authorization and execution switches in the gate cell. The new working SQLite is a one-time copy of v5, stored separately. On subsequent sessions, it resumes from that working SQLite, never recopying the seed.

In [1]:
# 1. Mount Drive (Colab only)
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; use locally mounted paths.')

Mounted at /content/drive


In [2]:
# 2. Dependencies and environment
import sys, subprocess
try:
    import transformers, tokenizers, numpy
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<5', 'tokenizers>=0.20', 'numpy>=1.26'])
import os, json, gzip, hashlib, sqlite3, time, shutil, platform, random, collections, struct
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
from transformers import AutoTokenizer
print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('Transformers:', __import__('transformers').__version__)

Python: 3.13.18
NumPy: 2.4.6
Transformers: 5.22.0


In [3]:
# Configuration — original production paths and parameters
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
OUT = ROOT / 'pretraining_production_v1_full'
FREEZE_MANIFEST = ROOT / 'freeze_v8_001/manifest.json'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
RECOVERY = OUT / 'recovery_audit_v4'
SEED_DB = RECOVERY / 'state_operational_v5.sqlite3'
WORK_DIR = RECOVERY / 'production_resume_v2'
DB = WORK_DIR / 'state_resume_v2.sqlite3'
CONFIG_PATH = OUT / 'run_config.json'
EXPECTED_TOTAL_CHUNKS = 21021
BLOCK_SIZE = 2048
SHARD_MAX_BLOCKS = 512
SPLIT_PERCENT = {'train':94,'validation':3,'test':3}
SPLIT_SEED = 'maia-v8-split-v1'
EOS_PER_DOCUMENT = True
SOURCE_NAMES = ['fineweb2_es','fineweb2_pt','fineweb_edu_en','wikipedia_en','wikipedia_es','wikipedia_pt']
LANGUAGE = {'fineweb2_es':'es','fineweb2_pt':'pt','fineweb_edu_en':'en','wikipedia_en':'en','wikipedia_es':'es','wikipedia_pt':'pt'}
LANGUAGE_TARGET = {'pt':0.40,'en':0.35,'es':0.25}
TARGET_TRAIN_EXPOSURES = 10_100_000_000
PRODUCTION_MODE = True
MAX_NEW_CHUNKS_PER_SESSION = 100  # Change to None for all remaining chunks in one session
PROGRESS_EVERY_NEW_CHUNKS = 10
VERIFY_EVERY_NEW_CHUNKS = 50
# No processing until both switches are deliberately set below.
assert OUT.is_dir() and SEED_DB.is_file() and CONFIG_PATH.is_file()
print('Seed:', SEED_DB)
print('Working DB:', DB)
print('Original DB untouched:', OUT/'state.sqlite3')

Seed: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full/recovery_audit_v4/state_operational_v5.sqlite3
Working DB: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full/recovery_audit_v4/production_resume_v2/state_resume_v2.sqlite3
Original DB untouched: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full/state.sqlite3


In [4]:
# 4. Validate source freeze and tokenizer fingerprint
assert FREEZE_MANIFEST.is_file(), f'Missing freeze: {FREEZE_MANIFEST}'
freeze_bytes = FREEZE_MANIFEST.read_bytes()
freeze = json.loads(freeze_bytes)
freeze_sha = hashlib.sha256(freeze_bytes).hexdigest()
assert TOKENIZER_DIR.is_dir(), f'Missing tokenizer: {TOKENIZER_DIR}'
TOK_FILES = ['merges.txt','tokenizer_config.json','vocab.json','tokenizer.json']
for filename in TOK_FILES:
    assert (TOKENIZER_DIR/filename).is_file(), f'Missing tokenizer file: {filename}'
file_hashes = {f:hashlib.sha256((TOKENIZER_DIR/f).read_bytes()).hexdigest() for f in TOK_FILES}
# These hashes are taken from the successful stratified pilot v2.
EXPECTED_HASHES = {
 'merges.txt':'b4b5c08bd27684fb3c6426ffe33adaaf7375340c733dad2f1d609d90767cef8a',
 'tokenizer_config.json':'a1cf74974f6d8a98aef2b7a2bb34084a6ad9d79c4e36e7280d369628149c7062',
 'vocab.json':'63b18c80a572b14e90de3529ac068b76ca49e449b7c4219eccbc05d6cee4cf6c',
 'tokenizer.json':'fd6cc1c4c1a487c5c4b45ee812762c471e5776ff1845fa48ad04c245ae4644a6'
}
assert file_hashes == EXPECTED_HASHES, 'Tokenizer files differ from validated pilot v2.'
tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True, use_fast=True)
assert len(tok) == 50257, f'Unexpected vocab size: {len(tok)}'
assert tok.eos_token_id == 2, f'Unexpected EOS: {tok.eos_token_id}'
assert tok.pad_token_id == 1, f'Unexpected PAD: {tok.pad_token_id}'
print('Freeze manifest SHA-256:', freeze_sha)
print('Tokenizer file hashes: MATCH pilot v2')
print('Historical tokenizer identity: NOT PROVEN')

Freeze manifest SHA-256: f72800af2ba0b176472d790e1781a41065cc7bc9b0439a0fcfb8c0cc26ee5846
Tokenizer file hashes: MATCH pilot v2
Historical tokenizer identity: NOT PROVEN


In [5]:
# Load the EXISTING immutable inventory, rather than rescan or recreate it.
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['production_mode'] is True
assert config['schema_version'] == '1.1-full'
assert config['block_size'] == BLOCK_SIZE
assert config['shard_max_blocks'] == SHARD_MAX_BLOCKS
assert config['split_percent'] == SPLIT_PERCENT
assert config['split_seed'] == SPLIT_SEED
assert config['eos_per_document'] == EOS_PER_DOCUMENT
assert config['tokenizer_file_sha256'] == file_hashes
assert config['freeze_manifest_sha256'] == freeze_sha
assert config['source_names'] == SOURCE_NAMES
INVENTORY_BY_PATH = {x['path']:x for x in config['selected_inventory']}
assert len(INVENTORY_BY_PATH) == EXPECTED_TOTAL_CHUNKS
selected = {s:[] for s in SOURCE_NAMES}
for record in config['selected_inventory']:
    selected[record['source']].append(Path(record['path']))
for s in SOURCE_NAMES:
    selected[s].sort(key=str)
selected_paths = [p for s in SOURCE_NAMES for p in selected[s]]
config_digest=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
print('Inventory:', len(selected_paths), 'chunks; configuration SHA:', config_digest)

Inventory: 21021 chunks; configuration SHA: 3162777c0bc575a9193cc643c8ea289e34716687a12734d45195c88e4149c8ef


In [6]:
# Initialize operational copy ONCE; subsequent sessions reuse it.
# Never copy the seed over an existing working database.
WORK_DIR.mkdir(parents=True,exist_ok=True)
if not DB.exists():
    assert not any(WORK_DIR.glob('state_resume_v2.sqlite3*')), 'Unexpected working SQLite sidecar: inspect manually'
    source=sqlite3.connect(SEED_DB.resolve().as_uri()+'?mode=ro',uri=True)
    try:
        assert source.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        assert source.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]==20371
        assert source.execute('SELECT COUNT(*) FROM seen').fetchone()[0]==9788069
        assert source.execute('SELECT COUNT(*) FROM shard').fetchone()[0]==60612
        temp=WORK_DIR/'state_resume_v2.sqlite3.initializing'
        assert not temp.exists(), 'Incomplete previous initialization; inspect manually'
        dest=sqlite3.connect(temp)
        try: source.backup(dest); dest.commit()
        finally: dest.close()
        check=sqlite3.connect(temp)
        try: assert check.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        finally: check.close()
        os.replace(temp,DB)
    finally: source.close()
    print('Created separate working DB from audited v5.')
else:
    print('Resuming existing working DB; seed NOT recopied.')
con=sqlite3.connect(DB,timeout=120)
con.execute('PRAGMA journal_mode=DELETE')
con.execute('PRAGMA synchronous=FULL')
con.execute('PRAGMA busy_timeout=120000')
assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
assert con.execute('SELECT COUNT(*) FROM recovery_provenance').fetchone()[0]==306
completed_paths={p for (p,) in con.execute('SELECT path FROM completed_chunk')}
assert completed_paths.issubset(INVENTORY_BY_PATH)
assert len(completed_paths)<=EXPECTED_TOTAL_CHUNKS
print('Completed:',len(completed_paths),'remaining:',EXPECTED_TOTAL_CHUNKS-len(completed_paths))

Created separate working DB from audited v5.
Completed: 20371 remaining: 650


In [7]:
# Original deterministic helpers; no expensive full-shard re-audit.
import unicodedata

def canonical_text(value):
    return unicodedata.normalize('NFC',value).strip()

def document_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def split_for_hash(h):
    bucket = int.from_bytes(hashlib.sha256((SPLIT_SEED + ':' + h).encode()).digest()[:8],'big') % 10000
    if bucket < SPLIT_PERCENT['train']*100: return 'train'
    if bucket < (SPLIT_PERCENT['train']+SPLIT_PERCENT['validation'])*100: return 'validation'
    return 'test'

def shard_file(source,split,index):
    return OUT/'shards'/source/split/f'{source}_{split}_{index:06d}.bin'


In [8]:
# 8. Process a chunk atomically, retaining tails between chunks
# Shards use little-endian uint32 token IDs, exactly BLOCK_SIZE IDs per block.
# Each (source,split) has its own token stream; streams never mix languages/splits.
# IMPORTANT: This cell only defines functions; the next cell performs processing.

def process_chunk(source,path):
    path = Path(path)
    row=con.execute('SELECT 1 FROM completed_chunk WHERE path=?',(str(path),)).fetchone()
    if row: return 'SKIP_COMPLETED'
    # Check source metadata against the immutable run config.
    record = INVENTORY_BY_PATH[str(path)]
    st=path.stat()
    assert (st.st_size,st.st_mtime_ns)==(record['size'],record['mtime_ns']), f'Source changed: {path}'
    # Prepare pending stream state in RAM. Existing SQLite state is not changed until commit.
    buffers={}
    for split in SPLIT_PERCENT:
        row=con.execute('SELECT ids FROM pending WHERE source=? AND split=?',(source,split)).fetchone()
        buffers[split]=np.frombuffer(row[0],dtype='<u4').astype('<u4').tolist() if row else []
    new_hashes=[]; local_hashes=set(); docs=dups=tokens=0
    new_shards=[]
    try:
        with gzip.open(path,'rt',encoding='utf-8') as stream:
            for line in stream:
                if not line.strip(): continue
                obj=json.loads(line)
                raw=obj.get('text')
                if not isinstance(raw,str): raise ValueError(f'Missing text in {path}')
                txt=canonical_text(raw)
                if not txt: continue
                h=document_hash(txt)
                if h in local_hashes or con.execute('SELECT 1 FROM seen WHERE hash=?',(h,)).fetchone():
                    dups+=1; continue
                local_hashes.add(h)
                split=split_for_hash(h)
                ids=tok.encode(txt,add_special_tokens=False)
                if EOS_PER_DOCUMENT: ids.append(tok.eos_token_id)
                assert all(0 <= t < len(tok) for t in ids)
                buffers[split].extend(ids)
                new_hashes.append((h,split)); docs+=1; tokens+=len(ids)
                if not PRODUCTION_MODE and docs+dups>=MAX_DOCS_PER_CHUNK_PILOT: break
        # Prepare full blocks for each split, and preserve incomplete tails.
        for split in SPLIT_PERCENT:
            buf=buffers[split]
            nfull=len(buf)//BLOCK_SIZE
            if nfull:
                previous=con.execute('SELECT COALESCE(MAX(shard_index),-1) FROM shard WHERE source=? AND split=?',(source,split)).fetchone()[0]
                for start_block in range(0,nfull,SHARD_MAX_BLOCKS):
                    take=min(SHARD_MAX_BLOCKS,nfull-start_block)
                    index=previous+1+sum(1 for x in new_shards if x[0]==source and x[1]==split)
                    target=shard_file(source,split,index)
                    assert not target.exists(), f'Uncommitted/orphan shard exists: {target}; investigate before resuming.'
                    target.parent.mkdir(parents=True,exist_ok=True)
                    arr=np.asarray(buf[start_block*BLOCK_SIZE:(start_block+take)*BLOCK_SIZE],dtype='<u4')
                    payload=arr.tobytes()
                    temp=target.with_suffix('.bin.tmp')
                    assert not temp.exists(), f'Orphan temporary shard: {temp}'
                    with temp.open('xb') as f:
                        f.write(payload); f.flush(); os.fsync(f.fileno())
                    os.replace(temp,target)
                    new_shards.append((source,split,index,str(target),take,hashlib.sha256(payload).hexdigest()))
            buffers[split]=buf[nfull*BLOCK_SIZE:]
            assert len(buffers[split])<BLOCK_SIZE
        with con:
            con.executemany('INSERT INTO seen(hash,split) VALUES(?,?)',new_hashes)
            con.executemany('INSERT INTO shard(source,split,shard_index,path,blocks,sha256) VALUES(?,?,?,?,?,?)',new_shards)
            for split in SPLIT_PERCENT:
                tail=np.asarray(buffers[split],dtype='<u4').tobytes()
                con.execute('INSERT INTO pending(source,split,ids) VALUES(?,?,?) ON CONFLICT(source,split) DO UPDATE SET ids=excluded.ids',(source,split,tail))
            con.execute('INSERT INTO completed_chunk(path,source,docs,duplicates,tokens,completed_utc) VALUES(?,?,?,?,?,?)',
                        (str(path),source,docs,dups,tokens,datetime.now(timezone.utc).isoformat()))
        return f'OK docs={docs} dup={dups} tokens={tokens} shards={len(new_shards)}'
    except Exception:
        # Never silently remove generated files: inspect orphans on recovery.
        raise

In [9]:
# Fast safety gate. No automatic quarantine or file movement.
committed = {str(Path(p).resolve()) for (p,) in con.execute('SELECT path FROM shard')}
shards_root=OUT/'shards'
orphans=[]
if shards_root.exists():
    for p in shards_root.rglob('*'):
        if p.is_file() and (p.name.endswith('.bin') or p.name.endswith('.bin.tmp')):
            if str(p.resolve()) not in committed: orphans.append(p)
assert not orphans, 'Orphan shards detected: inspect before continuing: '+str(orphans[:10])
assert con.execute('SELECT COUNT(*) FROM shard').fetchone()[0] >= 60612
for s in SOURCE_NAMES:
    for split in SPLIT_PERCENT:
        maximum=con.execute('SELECT COALESCE(MAX(shard_index),-1) FROM shard WHERE source=? AND split=?',(s,split)).fetchone()[0]
        target=shard_file(s,split,maximum+1)
        assert not target.exists(), 'Next shard output already exists: '+str(target)
print('FAST PREFLIGHT: PASS; no orphan shards; no output collisions')

FAST PREFLIGHT: PASS; no orphan shards; no output collisions


## Explicit authorization and run gate

The source folder is named `quarantine_pending_rights_review`. **Do not enable processing unless you have completed and documented the necessary rights review for all remaining selected chunks.** The gate below is intentionally off. The notebook does not move files out of quarantine or infer permission from file existence.

After authorizing, set `RIGHTS_REVIEW_APPROVED=True` and `RUN_PRODUCTION=True` in the next cell. This will write new shards in the existing `OUT/shards` tree and update **only** the separate working SQLite; it does not replace the old `state.sqlite3` or the v5 seed.

In [ ]:
RIGHTS_REVIEW_APPROVED = False  # Set True only after documented authorization
RUN_PRODUCTION = False           # Explicit opt-in
assert RIGHTS_REVIEW_APPROVED, 'Rights review is not approved: production remains disabled.'
assert RUN_PRODUCTION, 'Set RUN_PRODUCTION=True only when ready to process remaining chunks.'
print('Production authorized for this session.')

In [ ]:
# Process only missing chunks. Original process_chunk() handles one chunk per transaction.
import time
progress_dir=WORK_DIR/'progress'
progress_dir.mkdir(parents=True,exist_ok=True)
started=time.perf_counter(); new_chunks=0; new_tokens=0
for source in SOURCE_NAMES:
    for idx,path in enumerate(selected[source],1):
        if MAX_NEW_CHUNKS_PER_SESSION is not None and new_chunks>=MAX_NEW_CHUNKS_PER_SESSION: break
        if con.execute('SELECT 1 FROM completed_chunk WHERE path=?',(str(path),)).fetchone(): continue
        t0=time.perf_counter()
        result=process_chunk(source,path)
        elapsed=time.perf_counter()-t0
        assert result.startswith('OK '),result
        new_chunks+=1
        chunk_tokens=con.execute('SELECT tokens FROM completed_chunk WHERE path=?',(str(path),)).fetchone()[0]
        new_tokens+=chunk_tokens
        completed=con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]
        if new_chunks==1 or new_chunks%PROGRESS_EVERY_NEW_CHUNKS==0:
            print(f'[{completed:,}/{EXPECTED_TOTAL_CHUNKS:,}] {source} {idx} | {result} | {elapsed:.1f}s',flush=True)
        report={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'source':source,
                'path':str(path),'new_chunks_this_session':new_chunks,
                'new_tokens_this_session':new_tokens,'completed_chunks':completed,
                'remaining_chunks':EXPECTED_TOTAL_CHUNKS-completed,
                'working_db':str(DB),'seed_db':str(SEED_DB)}
        tmp=progress_dir/'latest.json.tmp';dest=progress_dir/'latest.json'
        tmp.write_text(json.dumps(report,indent=2),encoding='utf-8');os.replace(tmp,dest)
        if new_chunks%VERIFY_EVERY_NEW_CHUNKS==0:
            assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
    if MAX_NEW_CHUNKS_PER_SESSION is not None and new_chunks>=MAX_NEW_CHUNKS_PER_SESSION: break
completed=con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]
print('SESSION COMPLETE:',new_chunks,'new chunks;',completed,'/',EXPECTED_TOTAL_CHUNKS)
print('STATUS:','ALL_CHUNKS_DONE' if completed==EXPECTED_TOTAL_CHUNKS else 'PARTIAL — rerun notebook')

In [ ]:
# Compact final status and conservation checks (no full shard hash scan).
assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
seen_count=con.execute('SELECT COUNT(*) FROM seen').fetchone()[0]
docs_count=con.execute('SELECT COALESCE(SUM(docs),0) FROM completed_chunk').fetchone()[0]
assert seen_count==docs_count,(seen_count,docs_count)
assert con.execute('SELECT COUNT(*) FROM recovery_provenance').fetchone()[0]==306
completed=con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]
print('STATUS:', 'COMPLETE' if completed==EXPECTED_TOTAL_CHUNKS else 'RESUMABLE')
print('Completed:',completed,'/',EXPECTED_TOTAL_CHUNKS)
print('Seen:',seen_count,'Documents:',docs_count)
print('Working DB:',DB)
print('Original state.sqlite3 and audited v5 were not overwritten.')
con.close()

## Operational cautions

- **Single writer only.** Do not run the old notebook or another session simultaneously.
- The source folder is still under rights review; complete that review before enabling processing.
- If interrupted after shard rename but before SQLite commit, the next session intentionally stops on an orphan. Inspect it rather than deleting it automatically.
- The operational SQLite is stored on Google Drive and may be slower than local SSD; no unsafe bidirectional syncing is attempted.
- Once all 21,021 chunks are complete, perform the final integrity/conservation audit before using the corpus for training. This notebook does **not** train the model.
